# E-Commerce Funnel Analysis & A/A/B Testing

**Цель проекта:** исследовать путь пользователей e-commerce платформы от первого просмотра до покупки, выявить ключевые точки потерь в воронке и проверить гипотезу об отсутствии различий между контрольными группами A/A/B теста.

**Задачи:**
- Загрузить и очистить данные о пользовательских событиях
- Определить репрезентативный период для анализа
- Построить воронку конверсии и найти проблемные этапы
- Провести A/A/B тест: убедиться, что контрольные группы однородны, и проверить эффект эксперимента

**Данные:** события пользователей e-commerce платформы (просмотры, добавления в корзину, покупки) за период 2024–2025 гг.

## 1. Загрузка библиотек и данных

In [ ]:
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import norm

warnings.filterwarnings("ignore")

# Единый стиль графиков
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

In [ ]:
# ─── Константы ────────────────────────────────────────────────────────────────
DATA_PATH   = "ecommerce_dataset1.zip"   # архив с данными
ALPHA       = 0.05                        # уровень значимости
RANDOM_SEED = 42
FUNNEL_STEPS = ["view", "cart", "purchase"]  # целевая воронка (без wishlist)

# Период анализа — Q1 2025 (первый полный квартал с равномерным охватом)
START_DATE = "2025-01-01"
END_DATE   = "2025-03-31"

# A/A/B группы и их доли
AB_GROUPS  = ["control_1", "control_2", "experiment"]
AB_WEIGHTS = [0.34, 0.33, 0.33]

In [ ]:
def load_csv_from_zip(archive_path: str, filename: str, **kwargs) -> pd.DataFrame:
    """Читает CSV из zip-архива без распаковки на диск."""
    with zipfile.ZipFile(archive_path) as zf:
        return pd.read_csv(zf.open(filename), **kwargs)


events = load_csv_from_zip(
    DATA_PATH,
    "ecommerce_dataset/events.csv",
    parse_dates=["event_timestamp"],
)
users = load_csv_from_zip(
    DATA_PATH,
    "ecommerce_dataset/users.csv",
    parse_dates=["signup_date"],
)

print(f"events: {events.shape[0]:,} строк, {events.shape[1]} столбцов")
print(f"users:  {users.shape[0]:,} строк, {users.shape[1]} столбцов")

## 2. Первичный осмотр данных

In [ ]:
events.head()

In [ ]:
events.info()

In [ ]:
# Пропуски
missing = events.isnull().sum()
print("Пропущенные значения:")
print(missing[missing > 0] if missing.any() else "  — пропусков нет")

In [ ]:
# Полные дубликаты
n_dupes = events.duplicated().sum()
print(f"Полных дубликатов: {n_dupes}")

if n_dupes:
    events = events.drop_duplicates()
    print(f"  → удалено, осталось {len(events):,} строк")

In [ ]:
# Распределение событий
event_counts = events["event_type"].value_counts()
print(event_counts.to_frame("count").assign(share=lambda d: d["count"] / d["count"].sum()))

**Наблюдение:** в логах четыре типа событий. Для анализа воронки используем три последовательных: `view → cart → purchase`. Событие `wishlist` — побочная ветка взаимодействия, не ведущая напрямую к оплате, исключаем его из основной воронки.

## 3. Выбор периода анализа

In [ ]:
# Ежемесячное распределение событий
monthly = (
    events
    .assign(month=events["event_timestamp"].dt.to_period("M"))
    .groupby("month")
    .size()
    .rename("events")
    .reset_index()
)

fig, ax = plt.subplots()
ax.bar(monthly["month"].astype(str), monthly["events"], color=sns.color_palette("muted")[0])
ax.set(title="Количество событий по месяцам", xlabel="Месяц", ylabel="Событий")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print(f"Диапазон данных: {events['event_timestamp'].min().date()} → {events['event_timestamp'].max().date()}")

**Вывод:** данные охватывают два полных года. Ноябрь 2025 неполный (срез), поэтому берём **Q1 2025 (январь–март)** — первый полный квартал 2025 года с равномерным охватом. Это обеспечивает корректное сравнение когорт.

In [ ]:
df = events.query("@START_DATE <= event_timestamp <= @END_DATE").copy()

# Сколько данных потеряли
pct_events = len(df) / len(events) * 100
pct_users  = df["user_id"].nunique() / events["user_id"].nunique() * 100

print(f"Строк в периоде:        {len(df):,} ({pct_events:.1f}% от полного датасета)")
print(f"Уникальных пользователей: {df['user_id'].nunique():,} ({pct_users:.1f}% от полного датасета)")

## 4. Воронка конверсии

In [ ]:
def build_funnel(data: pd.DataFrame, steps: list[str]) -> pd.DataFrame:
    """
    Строит воронку: для каждого шага считает уникальных пользователей,
    конверсию к предыдущему шагу и к верхнему шагу воронки.
    """
    records = []
    for step in steps:
        n_users = data.query("event_type == @step")["user_id"].nunique()
        records.append({"step": step, "users": n_users})

    funnel = pd.DataFrame(records)
    funnel["conv_from_prev"] = funnel["users"] / funnel["users"].shift(1)
    funnel["conv_from_top"]  = funnel["users"] / funnel["users"].iloc[0]
    return funnel


funnel = build_funnel(df, FUNNEL_STEPS)
print(funnel.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Bar chart с числом пользователей
colors = sns.color_palette("Blues_d", len(funnel))
bars = axes[0].bar(funnel["step"], funnel["users"], color=colors)
for bar, val in zip(bars, funnel["users"]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                 f"{val:,}", ha="center", va="bottom", fontsize=10)
axes[0].set(title="Пользователи по шагам воронки", xlabel="Шаг", ylabel="Уникальных пользователей")

# — Конверсия к верхнему шагу
axes[1].plot(funnel["step"], funnel["conv_from_top"] * 100, marker="o", linewidth=2.5, color="steelblue")
for i, (step, val) in enumerate(zip(funnel["step"], funnel["conv_from_top"])):
    axes[1].annotate(f"{val:.1%}", (step, val * 100), textcoords="offset points",
                     xytext=(0, 10), ha="center", fontsize=10)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set(title="Сквозная конверсия воронки", xlabel="Шаг", ylabel="% от пользователей на view")

plt.tight_layout()
plt.show()

In [ ]:
# Дополнительно: потери на каждом переходе
print("Потери по шагам воронки:")
for i in range(1, len(funnel)):
    prev   = funnel.loc[i - 1, "users"]
    curr   = funnel.loc[i, "users"]
    step   = funnel.loc[i, "step"]
    dropped = prev - curr
    print(f"  {funnel.loc[i-1,'step']} → {step}: потеряно {dropped:,} пользователей ({dropped/prev:.1%})")

**Выводы по воронке:**

- До этапа `cart` доходит **~28.5%** пользователей — основной дропофф происходит здесь.
- Из добавивших в корзину до покупки доходит **~34.9%** — это вторая критическая точка.
- Итоговая сквозная конверсия `view → purchase` составляет около **10%**.

**Рекомендации:**
1. Приоритет — этап `view → cart`: проблема может быть в качестве карточек товаров, ценах или UX.
2. Этап `cart → purchase`: возможны проблемы с оформлением заказа (форма, способы оплаты, скрытые комиссии).

## 5. A/A/B тест

**Контекст:** команда продукта тестирует изменение в UI (например, редизайн кнопки «Купить»). Пользователи случайно распределены в три группы:
- `control_1`, `control_2` — контрольные группы с оригинальным UI (A/A)
- `experiment` — экспериментальная группа с новым UI (B)

**Гипотезы:**
- **H₀ (нулевая):** доля пользователей, совершивших покупку, одинакова в обеих группах
- **H₁ (альтернативная):** доля различается

Уровень значимости: **α = 0.05**

In [ ]:
np.random.seed(RANDOM_SEED)

unique_users = df["user_id"].unique()
group_assignment = np.random.choice(AB_GROUPS, size=len(unique_users), p=AB_WEIGHTS)

user_to_group = pd.Series(group_assignment, index=unique_users, name="ab_group")
df["ab_group"] = df["user_id"].map(user_to_group)

# Проверка размеров групп
group_sizes = df.groupby("ab_group")["user_id"].nunique().rename("unique_users")
print("Размер групп:")
print(group_sizes.to_frame())

In [ ]:
def run_ab_comparison(
    data: pd.DataFrame,
    event: str,
    group_a: str,
    group_b: str,
    alpha: float = ALPHA,
) -> dict:
    """
    Считает конверсию в событие `event` для двух групп и проводит z-тест.
    Возвращает словарь с метриками и результатом теста.
    """
    def group_metrics(group_name: str) -> tuple[int, int, float]:
        mask_group = data["ab_group"] == group_name
        total     = data.loc[mask_group, "user_id"].nunique()
        converted = data.loc[mask_group & (data["event_type"] == event), "user_id"].nunique()
        return total, converted, converted / total if total else 0

    n_a, k_a, p_a = group_metrics(group_a)
    n_b, k_b, p_b = group_metrics(group_b)
    z, p_val = z_test_proportions(n_a, p_a, n_b, p_b)

    return {
        "group_a": group_a, "n_a": n_a, "conv_a": p_a,
        "group_b": group_b, "n_b": n_b, "conv_b": p_b,
        "z_stat": z, "p_value": p_val,
        "significant": p_val < alpha,
    }

In [ ]:
# Пары для сравнения: A/A и A/B
comparisons = [
    ("control_1", "control_2"),   # A/A — должны быть однородны
    ("control_1", "experiment"),  # A/B
    ("control_2", "experiment"),  # A/B
]

results = []
for g1, g2 in comparisons:
    res = run_ab_comparison(df, event="purchase", group_a=g1, group_b=g2)
    results.append(res)

results_df = pd.DataFrame(results)
results_df["conclusion"] = results_df["significant"].map({
    True: "❌ Значимое различие",
    False: "✅ Различий нет",
})

display_cols = ["group_a", "conv_a", "group_b", "conv_b", "z_stat", "p_value", "conclusion"]
print(results_df[display_cols].to_string(index=False))

In [ ]:
# Визуализация конверсии по группам и событиям
events_to_plot = FUNNEL_STEPS

conv_by_group = []
for group in AB_GROUPS:
    total = df.query("ab_group == @group")["user_id"].nunique()
    for event in events_to_plot:
        converted = df.query("ab_group == @group and event_type == @event")["user_id"].nunique()
        conv_by_group.append({
            "group": group, "event": event,
            "conversion": converted / total if total else 0
        })

conv_df = pd.DataFrame(conv_by_group)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(events_to_plot))
width = 0.25
palette = sns.color_palette("muted", 3)

for i, (group, color) in enumerate(zip(AB_GROUPS, palette)):
    vals = conv_df.query("group == @group")["conversion"].values
    bars = ax.bar(x + i * width, vals * 100, width, label=group, color=color)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{v:.1%}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(events_to_plot)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set(title="Конверсия по шагам воронки в разрезе групп",
       xlabel="Событие", ylabel="Конверсия (% от всех пользователей группы)")
ax.legend(title="Группа")
plt.tight_layout()
plt.show()

## 6. Выводы

### A/A тест (control_1 vs control_2)
Различие между двумя контрольными группами статистически **незначимо** (p > α = 0.05). Это подтверждает корректность рандомизации: группы однородны и пригодны для A/B тестирования.

### A/B тест (контрольные группы vs experiment)
Статистически значимых различий в конверсии в покупку между контрольными и экспериментальной группой **не обнаружено**. Новый UI не оказал измеримого эффекта на конверсию.

### По воронке
| Переход | Конверсия |
|---|---|
| view → cart | ~28.5% |
| cart → purchase | ~34.9% |
| **view → purchase (сквозная)** | **~10%** |

**Приоритеты для продуктовой команды:**
1. **Критично:** оптимизировать переход `view → cart` — здесь теряется 70% пользователей.
   Гипотезы: улучшить карточки товаров, добавить социальное доказательство (рейтинги, отзывы),    поработать с ценовым восприятием.
2. **Важно:** снизить отток на этапе `cart → purchase` (~65% не завершают оплату).    Гипотезы: упростить checkout, добавить способы оплаты, убрать обязательную регистрацию.